# PA4 Task 1 — Colab launcher
Thin launcher (no experiment logic). Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**.
Reads data from `MyDrive/atml-pa4-data`, writes results to `MyDrive/atml-pa4-results/task1`.
Every (condition × eval) is its own file and is skipped on rerun, so after a disconnect just Run all again.

## 0. What to run
Edit only this cell.

In [ ]:
BRANCH= "task1-compress"
METHODS= ["baseline"]
EVALS= "wikitext2 c4_val hellaswag resources"

## 1. GPU and Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount("/content/drive")

## 2. Code, data, packages

In [ ]:
import os
os.environ["BRANCH"]= BRANCH
os.environ["METHODS"]= " ".join(METHODS)
os.environ["EVALS"]= EVALS

In [ ]:
%%bash
set -e
cd /content
rm -rf atml-pa4 atml-pa4-datasets
git clone -q -b "$BRANCH" https://github.com/Zeeba1705/atml-pa4.git
git clone -q https://github.com/AbDu11aHHH/atml-pa4-datasets.git
pip install -q transformers==4.57.1 huggingface_hub==0.36.2 "datasets>=3,<5" pyyaml

#data generated earlier (colab/setup_banks.ipynb), same lock as splits/data_lock
DATA=/content/drive/MyDrive/atml-pa4-data
cp -r "$DATA/public_locked" atml-pa4-datasets/data/
diff "$DATA/local_source_revisions.json" atml-pa4/splits/data_lock/local_source_revisions.json && echo "LOCK MATCHES"

echo "code at $(git -C atml-pa4 rev-parse --short HEAD) on $BRANCH"
python -c "import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))"

## 3. Tests (CPU, ~30 s) — stops here if anything is broken

In [ ]:
%%bash
set -e
cd /content/atml-pa4
python -m pytest tests -q 2>&1 | tail -3

## 4. Run conditions (log saved to Drive)

In [ ]:
%%bash
set -e
cd /content/atml-pa4
RES=/content/drive/MyDrive/atml-pa4-results/task1
mkdir -p "$RES/logs"
for m in $METHODS; do
  PYTHONUNBUFFERED=1 python -m task1.run_condition --method "$m" --evals $EVALS --data_root /content/atml-pa4-datasets --results_dir "$RES" --output_dir /content/outputs \
    2>&1 | grep --line-buffered -v -i "warn" | tee -a "$RES/logs/$m.log"
  test ${PIPESTATUS[0]} -eq 0
done

## 5. Summary (paste this output to Claude)

In [ ]:
import json, glob, os
RES= "/content/drive/MyDrive/atml-pa4-results/task1"
for m in METHODS:
    print(f"--- {m} ---")
    for e in EVALS.split():
        p= os.path.join(RES, m, f"{e}.json")
        if not os.path.exists(p):
            print(e, "MISSING")
            continue
        d= json.load(open(p))
        if e in ["wikitext2", "c4_val"]:
            print(f"{e}: ppl={d['ppl']:.4f} n={d['n_scored']} windows={d['n_windows']} time={d['wall_clock_seconds']:.0f}s")
        elif e == "hellaswag":
            print(f"hellaswag: acc={d['acc']:.4f} acc_norm={d['acc_norm']:.4f} n={d['n_examples']} time={d['wall_clock_seconds']:.0f}s")
        else:
            for k, v in d["latency"].items():
                print(f"{k}: median={v['median']:.1f} ms iqr={v['iqr']:.1f} peak_alloc={v['peak_allocated_bytes']/2**30:.2f} GiB")
            print("storage:", d["storage"])
    print("commit:", d["git_commit"], "gpu:", d["gpu_name"], "dtype:", d["dtype"], "sdpa:", d["sdpa"])